# PathEval 2D test set: data setup (Kaggle)

Downloads the 2D **test** split of [PathEval](https://huggingface.co/datasets/maghzal/PathEval)
(Aghzal et al., CVPR 2025), fixes inconsistent image paths, checks that every image exists, and writes
the team manifest `test_2D.jsonl` (format: `docs/README.md`).

**Settings:** Accelerator = None, Internet = On. Optional Kaggle secret `HF_TOKEN` avoids HuggingFace rate limits.
After *Save & Run All*, turn the Output into a Kaggle Dataset (e.g. `patheval-2d-test`).

The path-fixing logic mirrors `src/data.py` (`resolve_image_path`, `to_manifest_row`); it is inlined so this
notebook runs without cloning the repo.

In [ ]:
import os, json
from collections import Counter

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Using HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret; downloading anonymously (may be rate limited)")

from huggingface_hub import HfApi, hf_hub_download, snapshot_download

REPO = "maghzal/PathEval"
LOCAL = "/kaggle/working/patheval"

## 1. File list and test records

In [ ]:
repo_files = set(HfApi().list_repo_files(REPO, repo_type="dataset"))
print(len(repo_files), "files in repo")
test = json.load(open(hf_hub_download(REPO, "2D/test_dataset_2D.json", repo_type="dataset")))
print("Test examples:", len(test))
print(json.dumps(test[0], indent=2))

## 2. Fix image paths
Images live in first-digit subfolders (`combined_images/2/2383_2D.png`), but at least one record omits the
subfolder (`combined_images/2_2D.png`). Try the path as written, then with the subfolder; fail loudly otherwise.

In [ ]:
def resolve_image_path(image, dim="2D"):
    folder, name = image.rsplit("/", 1)
    for cand in (f"{dim}/{image}", f"{dim}/{folder}/{name[0]}/{name}"):
        if cand in repo_files:
            return cand
    raise FileNotFoundError(image)

resolved = [resolve_image_path(ex["image"]) for ex in test]
print(sum(r != f"2D/{ex['image']}" for r, ex in zip(resolved, test)), "paths needed the subfolder fix")

## 3. Download test images
Only `combined_images`: `combined_images_2` is byte-identical for all 1050 test images (checked with MD5).

In [ ]:
snapshot_download(REPO, repo_type="dataset", allow_patterns=resolved, local_dir=LOCAL)
missing = [p for p in resolved if not os.path.exists(f"{LOCAL}/{p}")]
print("Missing after download:", len(missing))
assert not missing

## 4. Sanity checks

In [ ]:
from PIL import Image
print("Labels:", Counter(ex["ground truth"] for ex in test))
scen = Counter(ex["scenario"] for ex in test)
print(f"{len(scen)} scenarios, sizes {sorted(set(scen.values()))}")
print("Image sizes:", Counter(Image.open(f"{LOCAL}/{p}").size for p in resolved))

## 5. Write the team manifest (`docs/README.md` contract)

In [ ]:
rows = [{
    "sample_id": str(ex["id"]),
    "split": "test",
    "dimension": "2D",
    "image_paths": [p],                       # one side-by-side image: Path 1 left, Path 2 right
    "scenario": ex["scenario"],
    "label": int(ex["ground truth"].split()[-1]),
    "descriptors": {"path1": ex["path 1 descriptors"], "path2": ex["path 2 descriptors"]},
} for ex, p in zip(test, resolved)]

with open(f"{LOCAL}/test_2D.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")
print(f"Wrote {len(rows)} rows to {LOCAL}/test_2D.jsonl")